In [1]:
# ── Librerías ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import janitor
import sqlalchemy as sa
import os
from pathlib import Path
from dotenv import load_dotenv

%matplotlib inline

# ── Opciones de visualización ─────────────────────────────────────────
# Desactivar notación científica
pd.set_option('display.float_format', '{:.3f}'.format)
np.set_printoptions(suppress=True)

# Ver todas las columnas al imprimir un DataFrame
pd.set_option('display.max_columns', None)

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

# ── Variables de entorno ──────────────────────────────────────────────
load_dotenv(RAIZ / '.env')

print('✅ Entorno listo')
print(f'   Raíz del proyecto: {RAIZ}')

✅ Entorno listo
   Raíz del proyecto: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes


In [2]:
# ── TAREA 1: Carga del dataset y separación X/y ───────────────────────
# Dataframe actual según copilot-instructions.md
RUTA_DATASET = ENTRENAMIENTO / '04_train_tablon_transformado.pkl'
df = pd.read_pickle(RUTA_DATASET)

# ── Separación X / y ────────────────────────────────────────────────
TARGET = 'contrata_fondos'
X = df.drop(columns=[TARGET])
y = df[TARGET]

# ── Checks básicos ───────────────────────────────────────────────────
print('✅ Dataset cargado')
print(f'   Shape df:      {df.shape}')
print(f'   Features (X):  {X.shape[1]}')
print(f'   Filas:         {X.shape[0]}')

print('\n── Distribución target ─────────────────')
print(y.value_counts(normalize=True).mul(100).round(2).rename('%'))

print('\n── NaN por feature (top 5) ────────────')
print(X.isna().sum().sort_values(ascending=False).head(5).rename('NaNs'))

print('\n── Tipos de las features ──────────────')
print(X.dtypes.value_counts().rename('n_features'))

✅ Dataset cargado
   Shape df:      (28015, 34)
   Features (X):  33
   Filas:         28015

── Distribución target ─────────────────
contrata_fondos
0   88.460
1   11.540
Name: %, dtype: float64

── NaN por feature (top 5) ────────────
trabajo_blue-collar     0
trabajo_entrepreneur    0
trabajo_housemaid       0
trabajo_management      0
trabajo_retired         0
Name: NaNs, dtype: int64

── Tipos de las features ──────────────
int8       24
float64     9
Name: n_features, dtype: int64


In [3]:
# ── TAREA 2: Checks mínimos de coherencia ─────────────────────────────
# 1) Columnas constantes (un único valor): no aportan nada a un modelo lineal
n_unico = X.nunique()
constantes = n_unico[n_unico <= 1].index.tolist()

# 2) Columnas duplicadas exactas (misma info en dos columnas)
masc_dup = X.T.duplicated()
duplicadas_exactas = X.columns[masc_dup].tolist()

# 3) Columnas no numéricas (deberían ser 0)
no_numericas = X.select_dtypes(exclude=['number']).columns.tolist()

print('── Check 1: columnas constantes ────────')
print(f'   Nº columnas con un único valor: {len(constantes)}')
if constantes:
    print(f'   -> {constantes}')

print('\n── Check 2: columnas duplicadas ───────')
print(f'   Nº columnas duplicadas exactas: {len(duplicadas_exactas)}')
if duplicadas_exactas:
    print(f'   -> {duplicadas_exactas}')

print('\n── Check 3: columnas no numéricas ─────')
print(f'   Nº columnas no numéricas: {len(no_numericas)}')
if no_numericas:
    print(f'   -> {no_numericas}')

print('\n✅ Checks mínimos finalizados')

── Check 1: columnas constantes ────────
   Nº columnas con un único valor: 1
   -> ['contactado_previamente']

── Check 2: columnas duplicadas ───────
   Nº columnas duplicadas exactas: 1
   -> ['prestamo_personal_unknown']

── Check 3: columnas no numéricas ─────
   Nº columnas no numéricas: 0

✅ Checks mínimos finalizados


In [4]:
# ── Diagnóstico de hallazgos del Check 1 y 2 ──────────────────────────
print('── Valor único de contactado_previamente ──')
print(f'   valor = {X["contactado_previamente"].iloc[0]}  '
      f'(conteo: {X["contactado_previamente"].value_counts().to_dict()})')

print('\n── ¿Con qué columna duplica prestamo_personal_unknown? ──')
for col in X.columns:
    if col == 'prestamo_personal_unknown':
        continue
    if X[col].equals(X['prestamo_personal_unknown']):
        print(f'   -> Duplica EXACTAMENTE con: {col}')

── Valor único de contactado_previamente ──
   valor = 1  (conteo: {1: 28015})

── ¿Con qué columna duplica prestamo_personal_unknown? ──
   -> Duplica EXACTAMENTE con: prestamo_hipotecario_unknown


In [5]:
# ── Limpieza aprobada: eliminar columnas degeneradas ──────────────────
COLUMNAS_A_ELIMINAR = ['contactado_previamente', 'prestamo_personal_unknown']

# No modificamos X original: trabajamos sobre una copia limpia
X_limpio = X.drop(columns=COLUMNAS_A_ELIMINAR)

print('✅ Columnas eliminadas (aprobado por usuario):')
for col in COLUMNAS_A_ELIMINAR:
    print(f'   - {col}')
print(f'\nFeatures para preselección: {X.shape[1]} → {X_limpio.shape[1]}')
print(f'Shape X_limpio: {X_limpio.shape}')

✅ Columnas eliminadas (aprobado por usuario):
   - contactado_previamente
   - prestamo_personal_unknown

Features para preselección: 33 → 31
Shape X_limpio: (28015, 31)


In [9]:
# ── TAREA 3: RFECV con L1 (config PRUEBA: 2 folds) ────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

# Modelo lineal L1: la penalización pone a 0 los coeficientes poco útiles
# API sklearn >= 1.8: l1_ratio=1 equivale a penalty='l1' (sin deprecación)
modelo_l1 = LogisticRegression(
    l1_ratio=1,          # 1 = regularización L1
    solver='liblinear',  # solver L1 rápido para clasificación binaria
    C=1.0,               # inversa de regularización; menor = penalización más fuerte
    max_iter=1000,
    random_state=42
)

# CV estratificado por target + métrica AUC (clases desbalanceadas)
# PRUEBA: solo 2 folds para validar rápido el flujo
cv_estrat = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

rfecv = RFECV(
    estimator=modelo_l1,
    step=1,
    cv=cv_estrat,
    scoring='roc_auc',
    min_features_to_select=1,
    n_jobs=-1
)

print('🔄 Ejecutando RFECV (modo prueba, 2 folds)...')
rfecv.fit(X_limpio, y)

# ── Resultados ────────────────────────────────────────────────────────
variables_rfecv = list(X_limpio.columns[rfecv.support_])
ranking_rfecv = pd.Series(rfecv.ranking_, index=X_limpio.columns, name='ranking')

if hasattr(rfecv, 'cv_results_'):
    scores_rfecv = rfecv.cv_results_['mean_test_score']
else:  # sklearn antiguo
    scores_rfecv = rfecv.grid_scores_

print(f'\n✅ RFECV completado')
print(f'   Nº óptimo de variables: {rfecv.n_features_} de {X_limpio.shape[1]}')
print(f'   AUC media CV (todas):   {scores_rfecv[0]:.4f}')
print(f'   AUC media CV (óptimo):  {scores_rfecv.max():.4f}')

print('\n── Variables SELECCIONADAS ───────────────')
for i, var in enumerate(variables_rfecv, 1):
    print(f'   {i:2d}. {var}')

print('\n── Variables ELIMINADAS (ranking > 1) ───')
eliminadas = ranking_rfecv[ranking_rfecv > 1].sort_values()
for var, rk in eliminadas.items():
    print(f'   - {var} (ranking {rk})')

🔄 Ejecutando RFECV (modo prueba, 2 folds)...

✅ RFECV completado
   Nº óptimo de variables: 24 de 31
   AUC media CV (todas):   0.5904
   AUC media CV (óptimo):  0.7713

── Variables SELECCIONADAS ───────────────
    1. trabajo_blue-collar
    2. trabajo_entrepreneur
    3. trabajo_management
    4. trabajo_retired
    5. trabajo_services
    6. trabajo_student
    7. trabajo_unknown
    8. estado_civil_single
    9. impago_unknown
   10. prestamo_hipotecario_unknown
   11. prestamo_hipotecario_yes
   12. prestamo_personal_yes
   13. canal_de_contacto_telephone
   14. resultado_campana_anterior_nonexistent
   15. resultado_campana_anterior_success
   16. mes_sin
   17. mes_cos
   18. formacion_oe_imp_ss
   19. num_dias_ultimo_contacto_imp_ss
   20. num_contactos_esta_campana_log_ss
   21. num_contactos_otras_campanas_log_ss
   22. edad_ss
   23. variacion_tasa_empleo_ss
   24. euribor3m_ss

── Variables ELIMINADAS (ranking > 1) ───
   - trabajo_housemaid (ranking 2)
   - trabajo_self-e

In [8]:
# ── Comprobación API sklearn: L1 sin penalty deprecado ────────────────
import sklearn
print(f'sklearn {sklearn.__version__}\n')

from sklearn.linear_model import LogisticRegression

# Muestra pequeña para validar combinaciones rápido
X_t = X_limpio.iloc[:2000]
y_t = y.iloc[:2000]

def probar(nombre, **kwargs):
    try:
        m = LogisticRegression(max_iter=500, random_state=42, **kwargs)
        m.fit(X_t, y_t)
        n_cero = int((np.abs(m.coef_) < 1e-8).sum())
        print(f'  OK  {nombre}  (coefs a 0: {n_cero}/{m.coef_.shape[1]})')
    except Exception as e:
        print(f'  FALLA {nombre} -> {type(e).__name__}: {str(e)[:130]}')

print('API nueva (sin penalty=...):')
probar('saga     + l1_ratio=1', solver='saga', l1_ratio=1)
probar('liblinear + l1_ratio=1', solver='liblinear', l1_ratio=1)
print('\nAPI vieja (referencia):')
probar('liblinear + penalty=l1', solver='liblinear', penalty='l1')

sklearn 1.9.0

API nueva (sin penalty=...):
  OK  saga     + l1_ratio=1  (coefs a 0: 7/31)
  OK  liblinear + l1_ratio=1  (coefs a 0: 6/31)

API vieja (referencia):
  OK  liblinear + penalty=l1  (coefs a 0: 6/31)


c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [10]:
# ── TAREA 4: Mutual Information (MI) ──────────────────────────────────
from sklearn.feature_selection import mutual_info_classif

# MI mide la relación (lineal o no) de CADA variable con la target,
# de forma independiente y sin usar modelo
mi_scores = mutual_info_classif(X_limpio, y, random_state=42)

serie_mi = pd.Series(mi_scores, index=X_limpio.columns, name='MI')
serie_mi = serie_mi.sort_values(ascending=False)

print('── Mutual Information por variable (ordenadas) ──')
print(serie_mi.round(4).to_string())

# ── Umbral para candidatas ──────────────────────────────────────────
# Variables con MI claramente positivo: por encima de la mediana de los positivos
mi_positivos = serie_mi[serie_mi > 0]
if len(mi_positivos) > 0:
    umbral_mi = float(mi_positivos.median())
    variables_mi = list(serie_mi[serie_mi >= umbral_mi].index)
else:
    umbral_mi = 0.0
    variables_mi = []

print(f'\n✅ MI calculado')
print(f'   Nº variables con MI > 0:      {len(mi_positivos)}')
print(f'   Umbral propuesto (mediana):   {umbral_mi:.4f}')
print(f'   Variables candidatas por MI:  {len(variables_mi)}')
print(variables_mi)

── Mutual Information por variable (ordenadas) ──
euribor3m_ss                             0.074
variacion_tasa_empleo_ss                 0.056
num_dias_ultimo_contacto_imp_ss          0.032
resultado_campana_anterior_success       0.030
mes_cos                                  0.024
num_contactos_otras_campanas_log_ss      0.021
resultado_campana_anterior_nonexistent   0.018
mes_sin                                  0.018
canal_de_contacto_telephone              0.018
edad_ss                                  0.014
estado_civil_married                     0.007
impago_unknown                           0.007
prestamo_hipotecario_yes                 0.004
trabajo_student                          0.004
num_contactos_esta_campana_log_ss        0.004
trabajo_retired                          0.003
formacion_oe_imp_ss                      0.003
trabajo_unknown                          0.002
trabajo_blue-collar                      0.002
prestamo_personal_yes                    0.002
trabajo_en

In [11]:
# ── TAREA 5: Permutation Importance (PI) ──────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

# Baseline para PI: Random Forest (relaciones no lineales; visión distinta
# a la del RFECV lineal). Se entrena y evalúa en partición train/test.
X_train_pi, X_test_pi, y_train_pi, y_test_pi = train_test_split(
    X_limpio, y, test_size=0.2, stratify=y, random_state=42
)

modelo_pi = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,          # limita profundidad para la prueba
    random_state=42,
    n_jobs=-1
)

print('🔄 Entrenando Random Forest baseline...')
modelo_pi.fit(X_train_pi, y_train_pi)

print('🔄 Calculando Permutation Importance (puede tardar)...')
pi = permutation_importance(
    modelo_pi, X_test_pi, y_test_pi,
    n_repeats=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

serie_pi = pd.Series(pi.importances_mean, index=X_limpio.columns, name='PI')
serie_pi = serie_pi.sort_values(ascending=False)

print('── Permutation Importance por variable (ordenadas) ──')
print(serie_pi.round(4).to_string())

# ── Umbral para candidatas ──────────────────────────────────────────
# Mismo criterio que en MI: por encima de la mediana de los positivos
pi_positivos = serie_pi[serie_pi > 0]
if len(pi_positivos) > 0:
    umbral_pi = float(pi_positivos.median())
    variables_pi = list(serie_pi[serie_pi >= umbral_pi].index)
else:
    umbral_pi = 0.0
    variables_pi = []

print(f'\n✅ PI calculado')
print(f'   Nº variables con PI > 0:      {len(pi_positivos)}')
print(f'   Umbral propuesto (mediana):   {umbral_pi:.4f}')
print(f'   Variables candidatas por PI:  {len(variables_pi)}')
print(variables_pi)

🔄 Entrenando Random Forest baseline...
🔄 Calculando Permutation Importance (puede tardar)...
── Permutation Importance por variable (ordenadas) ──
variacion_tasa_empleo_ss                  0.058
euribor3m_ss                              0.019
mes_sin                                   0.011
mes_cos                                   0.010
canal_de_contacto_telephone               0.008
edad_ss                                   0.004
resultado_campana_anterior_success        0.004
num_dias_ultimo_contacto_imp_ss           0.004
num_contactos_esta_campana_log_ss         0.003
trabajo_student                           0.003
num_contactos_otras_campanas_log_ss       0.002
resultado_campana_anterior_nonexistent    0.002
impago_unknown                            0.002
formacion_oe_imp_ss                       0.001
trabajo_unknown                           0.001
prestamo_personal_yes                     0.000
trabajo_technician                        0.000
prestamo_hipotecario_unknown         

In [12]:
# ── TAREA 6: Combinación de los 3 métodos ─────────────────────────────
# Cada variable recibe 1 punto por método que la haya seleccionado (0-3)
tabla_combinada = pd.DataFrame(index=X_limpio.columns)
tabla_combinada['RFECV_L1'] = tabla_combinada.index.isin(variables_rfecv).astype(int)
tabla_combinada['MI']       = tabla_combinada.index.isin(variables_mi).astype(int)
tabla_combinada['PI']       = tabla_combinada.index.isin(variables_pi).astype(int)
tabla_combinada['score']    = tabla_combinada.sum(axis=1)
tabla_combinada = tabla_combinada.sort_values(['score', 'RFECV_L1', 'MI', 'PI'],
                                              ascending=[False, False, False, False])

print('── Tabla combinada (1 = seleccionada por ese método) ──')
print(tabla_combinada.to_string())

print('\n── Resumen por score ──')
print(tabla_combinada['score'].value_counts().sort_index(ascending=False).rename('n_vars'))

# ── Escenarios posibles ─────────────────────────────────────────────
esc_fuerte     = tabla_combinada[tabla_combinada['score'] == 3].index.tolist()
esc_amplio     = tabla_combinada[tabla_combinada['score'] >= 2].index.tolist()
esc_candidatas = tabla_combinada[tabla_combinada['score'] >= 1].index.tolist()

print('\n── Escenarios disponibles ──')
print(f'   CONSENSO FUERTE  (score = 3): {len(esc_fuerte)} vars')
print(f'   CONSENSO AMPLIO  (score >= 2): {len(esc_amplio)} vars')
print(f'   TODAS CANDIDATAS (score >= 1): {len(esc_candidatas)} vars')

── Tabla combinada (1 = seleccionada por ese método) ──
                                        RFECV_L1  MI  PI  score
trabajo_student                                1   1   1      3
canal_de_contacto_telephone                    1   1   1      3
resultado_campana_anterior_success             1   1   1      3
mes_sin                                        1   1   1      3
mes_cos                                        1   1   1      3
num_dias_ultimo_contacto_imp_ss                1   1   1      3
edad_ss                                        1   1   1      3
variacion_tasa_empleo_ss                       1   1   1      3
euribor3m_ss                                   1   1   1      3
impago_unknown                                 1   1   0      2
prestamo_hipotecario_yes                       1   1   0      2
resultado_campana_anterior_nonexistent         1   1   0      2
num_contactos_otras_campanas_log_ss            1   1   0      2
num_contactos_esta_campana_log_ss              1

In [13]:
# ── TAREA 6b: Fijar escenario elegido (CONSENSO AMPLIO, score >= 2) ──
variables_supervisadas_final = esc_amplio.copy()

# Importancias supervisadas agregadas (score 0-3) para la desduplicación
importancias_supervisadas = tabla_combinada['score'].to_dict()

print('✅ Escenario CONSENSO AMPLIO aplicado (elegido por usuario)')
print(f'   Variables supervisadas finales: {len(variables_supervisadas_final)}\n')
for i, var in enumerate(variables_supervisadas_final, 1):
    print(f'   {i:2d}. {var}  (score {importancias_supervisadas[var]})')

✅ Escenario CONSENSO AMPLIO aplicado (elegido por usuario)
   Variables supervisadas finales: 14

    1. trabajo_student  (score 3)
    2. canal_de_contacto_telephone  (score 3)
    3. resultado_campana_anterior_success  (score 3)
    4. mes_sin  (score 3)
    5. mes_cos  (score 3)
    6. num_dias_ultimo_contacto_imp_ss  (score 3)
    7. edad_ss  (score 3)
    8. variacion_tasa_empleo_ss  (score 3)
    9. euribor3m_ss  (score 3)
   10. impago_unknown  (score 2)
   11. prestamo_hipotecario_yes  (score 2)
   12. resultado_campana_anterior_nonexistent  (score 2)
   13. num_contactos_otras_campanas_log_ss  (score 2)
   14. num_contactos_esta_campana_log_ss  (score 2)


In [14]:
# ── TAREA 7: Agrupar derivadas por variable original ──────────────────
# Mapeo variable original -> columnas derivadas (Diseño_Transformaciones.md)
agrupacion_por_madre = {
    'trabajo': ['trabajo_blue-collar', 'trabajo_entrepreneur', 'trabajo_housemaid',
                'trabajo_management', 'trabajo_retired', 'trabajo_self-employed',
                'trabajo_services', 'trabajo_student', 'trabajo_technician',
                'trabajo_unemployed', 'trabajo_unknown'],
    'estado_civil': ['estado_civil_married', 'estado_civil_single', 'estado_civil_unknown'],
    'impago': ['impago_unknown', 'impago_yes'],
    'prestamo_hipotecario': ['prestamo_hipotecario_unknown', 'prestamo_hipotecario_yes'],
    'prestamo_personal': ['prestamo_personal_unknown', 'prestamo_personal_yes'],
    'canal_de_contacto': ['canal_de_contacto_telephone'],
    'formacion': ['formacion_oe_imp_ss'],
    'mes': ['mes_sin', 'mes_cos'],
    'resultado_campana_anterior': ['resultado_campana_anterior_nonexistent',
                                   'resultado_campana_anterior_success'],
    'edad': ['edad_ss'],
    'num_contactos_esta_campana': ['num_contactos_esta_campana_log_ss'],
    'num_contactos_otras_campanas': ['num_contactos_otras_campanas_log_ss'],
    'variacion_tasa_empleo': ['variacion_tasa_empleo_ss'],
    'euribor3m': ['euribor3m_ss'],
    'num_dias_ultimo_contacto': ['contactado_previamente', 'num_dias_ultimo_contacto_imp_ss'],
}

# Filtrar: solo derivadas que han superado la selección supervisada
agrupacion_filtrada = {
    madre: [der for der in derivadas if der in variables_supervisadas_final]
    for madre, derivadas in agrupacion_por_madre.items()
}
agrupacion_filtrada = {k: v for k, v in agrupacion_filtrada.items() if len(v) > 0}

print('── Derivadas seleccionadas por variable original ──')
for madre, derivadas in agrupacion_filtrada.items():
    print(f'   {madre:28s} -> {len(derivadas)} derivada(s): {derivadas}')

── Derivadas seleccionadas por variable original ──
   trabajo                      -> 1 derivada(s): ['trabajo_student']
   impago                       -> 1 derivada(s): ['impago_unknown']
   prestamo_hipotecario         -> 1 derivada(s): ['prestamo_hipotecario_yes']
   canal_de_contacto            -> 1 derivada(s): ['canal_de_contacto_telephone']
   mes                          -> 2 derivada(s): ['mes_sin', 'mes_cos']
   resultado_campana_anterior   -> 2 derivada(s): ['resultado_campana_anterior_nonexistent', 'resultado_campana_anterior_success']
   edad                         -> 1 derivada(s): ['edad_ss']
   num_contactos_esta_campana   -> 1 derivada(s): ['num_contactos_esta_campana_log_ss']
   num_contactos_otras_campanas -> 1 derivada(s): ['num_contactos_otras_campanas_log_ss']
   variacion_tasa_empleo        -> 1 derivada(s): ['variacion_tasa_empleo_ss']
   euribor3m                    -> 1 derivada(s): ['euribor3m_ss']
   num_dias_ultimo_contacto     -> 1 derivada(s): ['num_di

In [15]:
# ── TAREA 8: Desduplicación automática por variable madre ─────────────
# Detecta el tipo de grupo de derivadas para decidir si se conservan
# todas (OHE, cíclicas) o si hay codificaciones alternativas que duplican
def detectar_tipo_grupo(derivadas, X):
    """Clasifica el grupo de derivadas de una variable original."""
    def es_binaria(col):
        return bool(X[col].dropna().isin([0, 1]).all())

    # Todas binarias -> OHE / flags binarios (conservar todas)
    if len(derivadas) > 1 and all(es_binaria(c) for c in derivadas):
        prefijos = set()
        for col in derivadas:
            partes = col.rsplit('_', 1)
            prefijos.add(partes[0] if len(partes) == 2 else col)
        if len(prefijos) == 1:
            return 'OHE'
        return 'Ambiguo'

    # Par cíclico sin/cos del mismo prefijo (mes_sin, mes_cos) -> conservar
    if len(derivadas) == 2:
        sufijos = {col.rsplit('_', 1)[-1] for col in derivadas}
        prefijos = {col.rsplit('_', 1)[0] for col in derivadas}
        if sufijos == {'sin', 'cos'} and len(prefijos) == 1:
            return 'Ciclicas'

    # Múltiples versiones escaladas/transformadas de la misma base -> desduplicar
    if len(derivadas) > 1 and all(col.endswith('_ss') for col in derivadas):
        return 'EncodingsNumericos'

    return 'Ambiguo'


# ── Aplicar lógica automática ─────────────────────────────────────────
variables_conservar = []
variables_eliminar = []
decisiones_automaticas = []
casos_ambiguos = []

for variable_madre, derivadas in agrupacion_filtrada.items():
    if len(derivadas) == 1:
        variables_conservar.extend(derivadas)
        decisiones_automaticas.append({
            'variable_madre': variable_madre, 'tipo': 'Unica',
            'conservadas': derivadas, 'eliminadas': [],
            'justificacion': 'Solo una derivada superó la selección supervisada'
        })
        continue

    tipo_grupo = detectar_tipo_grupo(derivadas, X_limpio)

    if tipo_grupo in ['OHE', 'Ciclicas']:
        variables_conservar.extend(derivadas)
        decisiones_automaticas.append({
            'variable_madre': variable_madre, 'tipo': tipo_grupo,
            'conservadas': derivadas, 'eliminadas': [],
            'justificacion': f'Grupo tipo {tipo_grupo}: se conservan TODAS'
        })
    elif tipo_grupo in ['EncodingsNumericos', 'TransformacionesEstadisticas']:
        mejor = max(derivadas, key=lambda x: importancias_supervisadas.get(x, 0))
        elim = [d for d in derivadas if d != mejor]
        variables_conservar.append(mejor)
        variables_eliminar.extend(elim)
        decisiones_automaticas.append({
            'variable_madre': variable_madre, 'tipo': tipo_grupo,
            'conservadas': [mejor], 'eliminadas': elim,
            'justificacion': 'Conservada la derivada de mayor importancia supervisada'
        })
    else:
        casos_ambiguos.append({'variable_madre': variable_madre, 'derivadas': derivadas})

# ── Resumen ───────────────────────────────────────────────────────────
print('── Decisiones automáticas de desduplicación ──')
print(f"{'Madre':28s} {'Tipo':20s} {'Conservadas':60s} Eliminadas")
for d in decisiones_automaticas:
    print(f"{d['variable_madre']:28s} {d['tipo']:20s} "
          f"{str(d['conservadas']):60s} {d['eliminadas']}")

print(f'\nVariables conservadas: {len(variables_conservar)}')
print(f'Variables eliminadas automáticamente: {len(variables_eliminar)}')
print(f'Casos ambiguos para revisión manual: {len(casos_ambiguos)}')
if casos_ambiguos:
    print(casos_ambiguos)

── Decisiones automáticas de desduplicación ──
Madre                        Tipo                 Conservadas                                                  Eliminadas
trabajo                      Unica                ['trabajo_student']                                          []
impago                       Unica                ['impago_unknown']                                           []
prestamo_hipotecario         Unica                ['prestamo_hipotecario_yes']                                 []
canal_de_contacto            Unica                ['canal_de_contacto_telephone']                              []
mes                          Ciclicas             ['mes_sin', 'mes_cos']                                       []
resultado_campana_anterior   OHE                  ['resultado_campana_anterior_nonexistent', 'resultado_campana_anterior_success'] []
edad                         Unica                ['edad_ss']                                                  []
num_contactos

In [16]:
# ── TAREA 9: Correlación entre variables de DIFERENTES originales ────
# Lista tras desduplicación por variable madre
variables_tras_depuracion_por_madre = variables_conservar.copy()

X_depurado = X_limpio[variables_tras_depuracion_por_madre]
corr = X_depurado.corr()

# Buscar pares con |correlación| alta entre originales distintos
UMBRAL_CORR = 0.85
pares_corr = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        valor = corr.iloc[i, j]
        if abs(valor) > UMBRAL_CORR:
            pares_corr.append((corr.columns[i], corr.columns[j], valor))

pares_corr_df = pd.DataFrame(pares_corr, columns=['var_A', 'var_B', 'correlacion'])
pares_corr_df['correlacion'] = pares_corr_df['correlacion'].abs()
pares_corr_df = pares_corr_df.sort_values('correlacion', ascending=False)

print(f'── Pares con |correlación| > {UMBRAL_CORR} (originales distintos) ──')
if len(pares_corr_df) > 0:
    print(pares_corr_df.to_string(index=False))
else:
    print('   Ninguno')

print(f'\nVariables tras desduplicación por madre: {len(variables_tras_depuracion_por_madre)}')

── Pares con |correlación| > 0.85 (originales distintos) ──
                                 var_A                               var_B  correlacion
resultado_campana_anterior_nonexistent num_contactos_otras_campanas_log_ss        0.958
    resultado_campana_anterior_success     num_dias_ultimo_contacto_imp_ss        0.949

Variables tras desduplicación por madre: 14


In [17]:
# ── TAREA 10: Eliminación por correlación (aprobada por usuario) ─────
variables_eliminar_por_correlacion = ['num_contactos_otras_campanas_log_ss',
                                      'num_dias_ultimo_contacto_imp_ss']

variables_preseleccionadas_final = [
    var for var in variables_tras_depuracion_por_madre
    if var not in variables_eliminar_por_correlacion
]

# Validación de dimensiones
X_preseleccion = X_limpio[variables_preseleccionadas_final]

print('✅ Eliminación por correlación aplicada (aprobada por usuario)')
print(f'   Variables eliminadas: {len(variables_eliminar_por_correlacion)}')
for var in variables_eliminar_por_correlacion:
    print(f'     - {var}')

print(f'\n   Variables tras depuración por madre: {len(variables_tras_depuracion_por_madre)}')
print(f'   Variables preseleccionadas finales:  {len(variables_preseleccionadas_final)}')
print(f'   Shape dataset preseleccionado:       {X_preseleccion.shape}')

print('\n── LISTA FINAL DE VARIABLES ──')
for i, var in enumerate(variables_preseleccionadas_final, 1):
    print(f'   {i:2d}. {var}')

print('\n── Resumen reducción total ──')
print(f'   Features iniciales (tablón):      {X.shape[1]}')
print(f'   Features preseleccionadas final:  {len(variables_preseleccionadas_final)}')

✅ Eliminación por correlación aplicada (aprobada por usuario)
   Variables eliminadas: 2
     - num_contactos_otras_campanas_log_ss
     - num_dias_ultimo_contacto_imp_ss

   Variables tras depuración por madre: 14
   Variables preseleccionadas finales:  12
   Shape dataset preseleccionado:       (28015, 12)

── LISTA FINAL DE VARIABLES ──
    1. trabajo_student
    2. impago_unknown
    3. prestamo_hipotecario_yes
    4. canal_de_contacto_telephone
    5. mes_sin
    6. mes_cos
    7. resultado_campana_anterior_nonexistent
    8. resultado_campana_anterior_success
    9. edad_ss
   10. num_contactos_esta_campana_log_ss
   11. variacion_tasa_empleo_ss
   12. euribor3m_ss

── Resumen reducción total ──
   Features iniciales (tablón):      33
   Features preseleccionadas final:  12


In [18]:
# ── TAREA 11: Guardado de artefactos finales ──────────────────────────
import os

# 1) Dataset preseleccionado (features seleccionadas + target)
df_preseleccion = X_limpio[variables_preseleccionadas_final].copy()
df_preseleccion[TARGET] = y

RUTA_DATASET_PRESEL = ENTRENAMIENTO / '05_train_tablon_preseleccion.pkl'
df_preseleccion.to_pickle(RUTA_DATASET_PRESEL)

# 2) Lista de variables preseleccionadas
RUTA_LISTA_VARS = RAIZ / '01_Documentos' / 'Variables_preseleccionadas.txt'
with open(RUTA_LISTA_VARS, 'w', encoding='utf-8') as f:
    for var in variables_preseleccionadas_final:
        f.write(var + '\n')

# 3) Informe de preselección en resultados
CARPETA_INFORME = RESULTADOS / 'Preseleccion'
os.makedirs(CARPETA_INFORME, exist_ok=True)
RUTA_INFORME = CARPETA_INFORME / 'Informe_Preseleccion_Variables.md'

# ── Tablas del informe ────────────────────────────────────────────────
filas_dedup = []
for d in decisiones_automaticas:
    conservadas = ', '.join(d['conservadas'])
    eliminadas = ', '.join(d['eliminadas']) if d['eliminadas'] else '—'
    filas_dedup.append(f"| {d['variable_madre']} | {d['tipo']} | {conservadas} | {eliminadas} |")
tabla_dedup = '\n'.join(filas_dedup)

filas_corr = []
for fila in pares_corr_df.itertuples():
    filas_corr.append(f"| {fila.var_A} | {fila.var_B} | {abs(fila.correlacion):.3f} |")
tabla_corr = '\n'.join(filas_corr) if filas_corr else '| — | — | — |'

filas_elim_corr = '\n'.join(f'| {v} |' for v in variables_eliminar_por_correlacion)

filas_finales = '\n'.join(f'{i}. {v}' for i, v in enumerate(variables_preseleccionadas_final, 1))

n_ini = X.shape[1]
n_fin = len(variables_preseleccionadas_final)
reduccion = (1 - n_fin / n_ini) * 100

informe = f"""# Informe de Preselección de Variables

**Fecha**: 2026-09-02
**Proyecto**: Clasificación binaria — scoring de probabilidad de contratación de fondos
**Target**: `{TARGET}`

## Resumen

| Métrica | Valor |
|---|---|
| Variables iniciales (tablón transformado) | {n_ini} |
| Variables preseleccionadas finales | {n_fin} |
| Reducción | {reduccion:.1f} % |

## Métodos utilizados (modo comparativo)

Se ejecutaron 3 métodos supervisados y sus resultados se combinaron con un
sistema de puntuación (0-3 puntos, 1 por método que selecciona la variable).

### 1) RFECV con regularización L1 (método principal)

- Modelo: Regresión Logística con regularización L1
  (`solver='liblinear'`, `l1_ratio=1`, `C=1.0`).
- Validación: StratifiedKFold estratificado por target (config de prueba: 2 folds).
- Métrica: ROC-AUC.
- Nº óptimo de variables: **24 de 31**.
- AUC media CV con todas las variables: 0.5904.
- AUC media CV con el conjunto óptimo: 0.7713.

### 2) Mutual Information (MI)

- Método sin modelo; mide la relación individual de cada variable con el target.
- Umbral aplicado: mediana de los MI positivos = {umbral_mi:.4f}.
- Variables candidatas por MI: {len(variables_mi)}.

### 3) Permutation Importance (PI)

- Baseline: Random Forest (`n_estimators=150`, `max_depth=8`), partición 80/20.
- Umbral aplicado: mediana de los PI positivos = {umbral_pi:.4f}.
- Variables candidatas por PI: {len(variables_pi)}.

## Sistema de combinación y escenario elegido

Cada variable recibe 1 punto por método que la haya seleccionado. El usuario
eligió el escenario **CONSENSO AMPLIO (score >= 2)**, que dejó
**{len(variables_supervisadas_final)} variables** tras la fase supervisada.

## Decisiones de desduplicación automática (por variable original)

Lógica automática aplicada sobre los grupos de derivadas de cada variable
original, conservando completos los grupos OHE/cíclicos y eliminando solo
codificaciones redundantes seguras.

| Variable original | Tipo grupo | Conservadas | Eliminadas |
|---|---|---|---|
{tabla_dedup}

**Estadísticas de desduplicación**:
- Variables tras métodos supervisados: {len(variables_supervisadas_final)}.
- Grupos OHE/cíclicos conservados completos: {sum(1 for d in decisiones_automaticas if d['tipo'] in ['OHE', 'Ciclicas'])}.
- Grupos desduplicados (encodings alternativos): {sum(1 for d in decisiones_automaticas if d['tipo'] not in ['OHE', 'Ciclicas', 'Unica'])}.
- Variables eliminadas en esta fase: {len(variables_eliminar)}.
- Casos ambiguos (revisión manual): {len(casos_ambiguos)}.

## Eliminación por correlación entre originales distintos

Umbral |correlación| > {UMBRAL_CORR}. Pares detectados:

| Variable A | Variable B | |Correlación| |
|---|---|---:|
{tabla_corr}

Decisión del usuario (conservar la señal directa de campaña previa):

| Variables eliminadas por correlación |
|---|
{filas_elim_corr}

## Variables preseleccionadas finales ({n_fin})

{filas_finales}

## Recomendaciones para la modelización

- Con **12 variables** el dataset queda muy manejable y sin colinealidades
  fuertes entre originales distintos (adecuado para modelos lineales).
- La configuración de esta ejecución fue de **prueba** (RFECV con 2 folds,
  RF baseline limitado). Si se desea robustez máxima, re-ejecutar con más
  folds y repetir PI con más repeticiones antes de la modelización definitiva.
- Recordar que `contrata_fondos` está **desbalanceada** (~11.5% positivos):
  valorar estrategias de balanceo o métricas tipo AUC/PR en la modelización.
"""

with open(RUTA_INFORME, 'w', encoding='utf-8') as f:
    f.write(informe)

# ── Validación del guardado ──────────────────────────────────────────
print('✅ Artefactos guardados:')
print(f'   Dataset : {RUTA_DATASET_PRESEL}  ({df_preseleccion.shape})')
print(f'   Lista   : {RUTA_LISTA_VARS}')
print(f'   Informe : {RUTA_INFORME}')
print(f'\n   Existencia:')
print(f'   - {RUTA_DATASET_PRESEL.exists()}  {RUTA_DATASET_PRESEL}')
print(f'   - {RUTA_LISTA_VARS.exists()}  {RUTA_LISTA_VARS}')
print(f'   - {RUTA_INFORME.exists()}  {RUTA_INFORME}')

print('\n── Contenido Variables_preseleccionadas.txt ──')
print(open(RUTA_LISTA_VARS, encoding='utf-8').read())

✅ Artefactos guardados:
   Dataset : c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\02_datos\03_Entrenamiento\05_train_tablon_preseleccion.pkl  ((28015, 13))
   Lista   : c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\01_Documentos\Variables_preseleccionadas.txt
   Informe : c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\06_resultados\Preseleccion\Informe_Preseleccion_Variables.md

   Existencia:
   - True  c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\02_datos\03_Entrenamiento\05_train_tablon_preseleccion.pkl
   - True  c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\01_Documentos\Variables_preseleccionadas.txt
   - True  c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\06_resultados\Preseleccion\Informe_Preseleccion_Variables.md

── Conte

In [19]:
# ── TAREA 12: Salida df.info() para actualizar copilot-instructions ──
print('── Estructura del dataframe preseleccionado ──')
df_preseleccion.info()

── Estructura del dataframe preseleccionado ──
<class 'pandas.DataFrame'>
RangeIndex: 28015 entries, 0 to 28014
Data columns (total 13 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   trabajo_student                         28015 non-null  int8   
 1   impago_unknown                          28015 non-null  int8   
 2   prestamo_hipotecario_yes                28015 non-null  int8   
 3   canal_de_contacto_telephone             28015 non-null  int8   
 4   mes_sin                                 28015 non-null  float64
 5   mes_cos                                 28015 non-null  float64
 6   resultado_campana_anterior_nonexistent  28015 non-null  int8   
 7   resultado_campana_anterior_success      28015 non-null  int8   
 8   edad_ss                                 28015 non-null  float64
 9   num_contactos_esta_campana_log_ss       28015 non-null  float64
 10  variacion_tasa_empleo_